In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the CSV file
food_deli = os.path.join(path, 'Q1_data.csv')
df_food_deli = pd.read_csv(food_deli)



In [ ]:
# Task 2: Write your code here:
print(f"Shape: {df_food_deli.shape}")
df_food_deli.head()

In [ ]:
# Task 3: Write your code here:
df_food_deli.info()

In [ ]:
# Task 4: Write your code here:
df_food_deli.describe()

In [ ]:

# Task 5: Write your code here:
# Target distribution
print(f"Legendary: {df_food_deli['Delivery_Time'].sum()}")
print(f"Normal: {(df_food_deli['Delivery_Time'] == 0).sum()}")

# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df_food_deli, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
# Define stat columns
# Drop the 'Order_ID' column from the data
stat_cols = [ 'Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type','Preparation_Time_min', 'Courier_Experience_yrs','Delivery_Time']

# Drop rows with missing stat values
df_clean = df_food_deli.dropna(subset=stat_cols).copy()
print(f"Shape after cleaning: {df_clean.shape}")

In [ ]:
print(f"Shape before cleaning: {df_clean.shape}")

In [ ]:

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_clean)

In [ ]:
# Task 2: Write your code here:
#Handle missing values appropriately
#df_clean = df_food_deli.dropna(subset=stat_cols).copy()
#print(f"Shape after cleaning: {df_clean.shape}")
#print("Missing values remaining:", df_clean.isnull().sum().sum())
for col in ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']:
    df_clean[col] = df_clean[col].fillna('unknown')

print("Missing values remaining:", df_clean.isnull().sum().sum())


In [ ]:
# Task 3: Write your code here:
# Check and remove duplicates if any exist
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

#dropduplicates()


In [ ]:
# Task 4: Write your code here: Encode categorical variables if needed (Bonus if used One Hot Encoding)
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here: Apply feature scaling for all features (Use StandardScaler)
# Scale features - fit on train, transform both

from sklearn.preprocessing import MinMaxScaler
features = df_clean.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET
scaler = StandardScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])
df_clean.head()


In [ ]:
# Task 6: Write your code here: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)


In [ ]:
#X = df_clean.drop("delivery_time", axis=1).astype(float)
#y = food_deli['delivery_time'].astype(float)

# Define features and target
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

  # Validate
  y_pred = np.dot(X_test.values, theta)

  # Calculate evaluation metrics
  mse = sklearn_mse(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_losses.append(losses)
  lr_mse.append(mse)
  lr_rmse.append(rmse)
  lr_r2.append(r2)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

lr_losses = []
lr_mse = []
# Task 2,3,4,5: Write your code here:
# Mean Squared Error in NumPy
def mean_squared_error(y, y_hat):  # ما انتبهت انه MAE  الى اخر الوقت
  return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)


n_splits = 5  # K=5 Folds
# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200)
}
average_losses = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Task 1: Write your code here:   #   هنا ما امداني اصحح الغلط الي فوق  باقي 5 دقايق
coeffs = {}

coeffs['Lasso'] = models['LASSO Regression'].coef_
coeffs['Ridge'] = models['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here: